In [ ]:
!pip install opencv-python Pillow numpy pandas matplotlib scikit-learn tqdm ImageHash reportlab -q

In [ ]:
import os

dataset_path = '/kaggle/input/datasets/alimoumate/medical-imaging-pipeline/chestXray/chest-xray'

if os.path.exists(dataset_path):
    print(f"Dataset trouvé : {dataset_path}")
    !ls -la "{dataset_path}" | head -10
else:
    print(f"Chemin non trouvé : {dataset_path}")

In [ ]:
import shutil

# Dossiers à exporter
dossiers = [
    "/kaggle/working/ensemble_results"
]

zip_path = "/kaggle/working/results"

# Dossier temporaire
temp_dir = "/kaggle/working/export_results_all"

if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)

os.makedirs(temp_dir)

for dossier in dossiers:

    if os.path.exists(dossier):

        modele = os.path.basename(
            os.path.dirname(dossier)
        )

        type_dossier = os.path.basename(dossier)

        destination = os.path.join(
            temp_dir,
            modele,
            type_dossier
        )

        os.makedirs(
            os.path.dirname(destination),
            exist_ok=True
        )

        # Copier
        shutil.copytree(
            dossier,
            destination,
            dirs_exist_ok=True
        )

        print(f"Ajouté : {dossier}")

    else:
        print(f"Introuvable : {dossier}")


# Créer le ZIP
shutil.make_archive(
    zip_path,
    "zip",
    temp_dir
)

print("\n" + "=" * 60)
print("ZIP CRÉÉ AVEC SUCCÈS")
print("=" * 60)
print(f"{zip_path}.zip")

In [ ]:
print("\nLancement du test...")

!python /kaggle/input/datasets/alimoumate/medical-imaging-pipeline/training/main.py \
      --image-dir /kaggle/input/datasets/alexsanderlindolfo/raio-x-de-doenas-pulmonares-2-experimento/dataset_teste3/NORMAL \
      --checkpoints /kaggle/input/datasets/alimoumate/medical-imaging-pipeline/results_finetuned/training_results/finetuned/densenet121_best.pt /kaggle/input/datasets/alimoumate/medical-imaging-pipeline/results_finetuned/training_results/finetuned/efficientnet_b0_best.pt /kaggle/input/datasets/alimoumate/medical-imaging-pipeline/results_finetuned/training_results/finetuned/efficientnet_b2_best.pt /kaggle/input/datasets/alimoumate/medical-imaging-pipeline/results_finetuned/training_results/finetuned/resnet101_best.pt /kaggle/input/datasets/alimoumate/medical-imaging-pipeline/results_finetuned/training_results/finetuned/resnet50_best.pt \
      --output ./infer_out/predictions.csv

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

PRED_CSV = "/kaggle/working/infer_out/predictions.csv"
OUT_DIR = "/kaggle/working/prediction_stats"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(PRED_CSV)

print("Colonnes disponibles :")
print(df.columns.tolist())
print("\nAperçu :")
print(df.head())

has_true = "true_label" in df.columns
has_conf = "confidence" in df.columns

proba_cols = [c for c in df.columns if c.startswith("proba_")]
has_proba = len(proba_cols) > 0

print("\nRésumé du CSV :")
print(f"- Nombre d'images : {len(df)}")
print(f"- Labels vrais présents : {has_true}")
print(f"- Confiance présente : {has_conf}")
print(f"- Colonnes de probabilités : {len(proba_cols)}")

summary = {
    "n_samples": int(len(df)),
    "has_true_labels": bool(has_true),
    "has_confidence": bool(has_conf),
    "n_proba_columns": int(len(proba_cols)),
}

if has_true:
    y_true = df["true_label"].astype(str)
    y_pred = df["pred_label"].astype(str)

    summary["accuracy"] = float(accuracy_score(y_true, y_pred))
    summary["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    summary["f1_macro"] = float(f1_score(y_true, y_pred, average="macro"))
    summary["f1_weighted"] = float(f1_score(y_true, y_pred, average="weighted"))
    summary["precision_macro"] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    summary["recall_macro"] = float(recall_score(y_true, y_pred, average="macro", zero_division=0))

    print("\n=== MÉTRIQUES GLOBALES ===")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k:20s}: {v:.4f}")
        else:
            print(f"{k:20s}: {v}")

    print("\n=== RAPPORT DE CLASSIFICATION ===")
    report = classification_report(y_true, y_pred, digits=4, zero_division=0)
    print(report)

    # Sauvegarde texte
    with open(os.path.join(OUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(report)

    # Rapport détaillé en dataframe
    report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(os.path.join(OUT_DIR, "classification_report.csv"))
    print("\nRapport de classification sauvegardé.")

if has_true:
    labels = sorted(list(set(df["true_label"].astype(str).unique()) | set(df["pred_label"].astype(str).unique())))
    cm = confusion_matrix(df["true_label"].astype(str), df["pred_label"].astype(str), labels=labels)

    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Matrice de confusion")
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Prédit")
    plt.ylabel("Vrai")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
    plt.show()

    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    cm_df.to_csv(os.path.join(OUT_DIR, "confusion_matrix.csv"))
    print("\nMatrice de confusion sauvegardée.")

if has_true:
    df["correct"] = (df["true_label"].astype(str) == df["pred_label"].astype(str)).astype(int)
    error_df = df[df["correct"] == 0].copy()
    correct_df = df[df["correct"] == 1].copy()

    print("\n=== ERREURS ===")
    print(f"Nombre d'erreurs : {len(error_df)}")
    print(f"Taux d'erreur   : {len(error_df) / len(df):.4f}")

    error_df.to_csv(os.path.join(OUT_DIR, "misclassified_samples.csv"), index=False)
    correct_df.to_csv(os.path.join(OUT_DIR, "correct_samples.csv"), index=False)

    if len(error_df) > 0:
        print("\nQuelques erreurs :")
        cols_to_show = [c for c in ["filepath", "true_label", "pred_label", "confidence"] if c in error_df.columns]
        print(error_df[cols_to_show].head(20).to_string(index=False))

if has_conf:
    print("\n=== CONFIANCE ===")
    print(df["confidence"].describe().to_string())

    plt.figure(figsize=(8, 5))
    plt.hist(df["confidence"].dropna(), bins=20)
    plt.title("Distribution des confiances")
    plt.xlabel("Confidence")
    plt.ylabel("Nombre")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "confidence_distribution.png"), dpi=200, bbox_inches="tight")
    plt.show()

if has_true and has_conf:
    # Confiance moyenne correcte / incorrecte
    print("\n=== CONFIANCE PAR CORRECTION ===")
    conf_summary = df.groupby("correct")["confidence"].agg(["count", "mean", "min", "max"]).reset_index()
    conf_summary["correct"] = conf_summary["correct"].map({0: "incorrect", 1: "correct"})
    print(conf_summary.to_string(index=False))
    conf_summary.to_csv(os.path.join(OUT_DIR, "confidence_by_correctness.csv"), index=False)

    # Courbe simple : accuracy par bins de confiance
    bins = np.linspace(0, 1, 11)
    df["conf_bin"] = pd.cut(df["confidence"], bins=bins, include_lowest=True)
    bin_stats = df.groupby("conf_bin", observed=False)["correct"].agg(["count", "mean"]).reset_index()
    bin_stats.rename(columns={"mean": "accuracy"}, inplace=True)
    print("\n=== ACCURACY PAR BIN DE CONFIANCE ===")
    print(bin_stats.to_string(index=False))

    bin_stats.to_csv(os.path.join(OUT_DIR, "accuracy_by_confidence_bins.csv"), index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(len(bin_stats)), bin_stats["accuracy"].values, marker="o")
    plt.xticks(range(len(bin_stats)), [str(x) for x in bin_stats["conf_bin"]], rotation=45, ha="right")
    plt.ylim(0, 1.05)
    plt.title("Accuracy selon les bins de confiance")
    plt.xlabel("Bin de confiance")
    plt.ylabel("Accuracy")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "accuracy_by_confidence_bins.png"), dpi=200, bbox_inches="tight")
    plt.show()

if has_true:
    class_stats = (
        df.groupby("true_label")
        .apply(lambda g: pd.Series({
            "support": len(g),
            "accuracy_class": (g["true_label"].astype(str) == g["pred_label"].astype(str)).mean(),
            "mean_confidence": g["confidence"].mean() if "confidence" in g.columns else np.nan,
            "error_rate": 1 - (g["true_label"].astype(str) == g["pred_label"].astype(str)).mean(),
        }))
        .reset_index()
        .sort_values("support", ascending=False)
    )

    print("\n=== STATISTIQUES PAR CLASSE VRAIE ===")
    print(class_stats.to_string(index=False))
    class_stats.to_csv(os.path.join(OUT_DIR, "class_stats.csv"), index=False)

if has_true:
    confusions = (
        df[df["true_label"].astype(str) != df["pred_label"].astype(str)]
        .groupby(["true_label", "pred_label"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    print("\n=== TOP CONFUSIONS ===")
    print(confusions.head(20).to_string(index=False))
    confusions.to_csv(os.path.join(OUT_DIR, "top_confusions.csv"), index=False)

if has_proba:
    print("\n=== ANALYSE DES PROBABILITÉS ===")
    proba_summary = df[proba_cols].describe().transpose()
    print(proba_summary.to_string())
    proba_summary.to_csv(os.path.join(OUT_DIR, "proba_summary.csv"))

with open(os.path.join(OUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\nTous les résultats ont été sauvegardés dans : {OUT_DIR}")